# DL30 — 手書き数字(optdigits)を多層ニューラルネットで分類・可視化

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ilu-h-bando/dl-tutorial/blob/main/DL30_optdigits_visual.ipynb)

NumPy だけで **7層の多層ニューラルネットワーク（MLP）** を実装し、UCI の手書き数字データ
**optdigits（8×8, 64次元）** を 0〜9 に分類します。順伝播・逆伝播・ドロップアウト・
He初期化まで、フレームワークを使わず自前で書いています。

DL01（XOR）で学んだ「順伝播→誤差→逆伝播→更新」を、**層を深くした本格的なネットワーク**に
スケールさせたものです。このノートの主役は **可視化** で、深い層で何が起きるかを絵で体感します。

- 学習データ・評価結果のサンプル画像（誤分類は色を変えて表示）
- 1枚の数字が各層を通るときの活性の様子（forward の可視化）
- 層ごとの勾配の大きさ（**勾配消失**が起きていないか一目で分かる棒グラフ）
- 混同行列と、クラスごとの precision / recall / F1

> ねらい：層を深くすると何が難しくなるのか（勾配消失など）を、数式ではなく「絵」で体感すること。

## セットアップ（初回のみ・Colabの場合）

このリポジトリは**プライベート**なので、Colab から中の `optdigits/` データを読むには
あなたの **GitHub Personal Access Token (PAT)** が必要です。

1. GitHub で PAT を発行（`repo` の読み取り権限があれば十分）
2. Colab 画面左の **🔑（シークレット）** アイコンをクリック
3. 「新しいシークレットを追加」→ **名前: `GH_TOKEN`**、値に発行した PAT を貼り付け
4. そのシークレットの「ノートブックからのアクセス」を **ON**
5. 次のセルの `GH_OWNER` / `GH_REPO` / `GH_BRANCH` が正しいか確認して実行

> 手元の Jupyter で実行する場合は PAT 不要です（`optdigits/` が隣にある前提で、次のセルは何もしません）。
> Colab には `numpy` / `matplotlib` が最初から入っているので、追加インストールは不要です。

In [ ]:
# =========================================================
# データ取得（Google Colab 対応セル）
#   ローカルの Jupyter ではそのまま次のセルに進めます。
#   Colab では GitHub のプライベートリポジトリから optdigits/ を取得します。
# =========================================================
import os

# ↓↓↓ 自分のリポジトリに合わせて変更してください ↓↓↓
GH_OWNER  = "ilu-h-bando"
GH_REPO   = "dl-tutorial"
GH_BRANCH = "main"
# ↑↑↑ ここまで ↑↑↑

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _in_colab():
    # プライベートリポジトリなので Personal Access Token (PAT) が必要です。
    # 事前に Colab 左の 🔑（シークレット）から、名前 GH_TOKEN で PAT を登録してください。
    from google.colab import userdata
    try:
        token = userdata.get("GH_TOKEN")
    except Exception:
        token = None

    if not token:
        raise RuntimeError(
            "Colab のシークレットに GH_TOKEN が設定されていません。\n"
            "左サイドバーの 🔑 アイコン → 『新しいシークレットを追加』→ 名前: GH_TOKEN、"
            "値: あなたの GitHub Personal Access Token（repo 読み取り権限）を登録し、"
            "このノートブックからのアクセスを ON にしてから再実行してください。"
        )

    if not os.path.isdir(GH_REPO):
        !git clone --depth 1 --branch {GH_BRANCH} https://{token}@github.com/{GH_OWNER}/{GH_REPO}.git
    # 既存コードの DATA_DIR="optdigits" をそのまま使えるよう、リポジトリ直下へ移動
    os.chdir(GH_REPO)

print("cwd            =", os.getcwd())
print("optdigits あり =", os.path.isdir("optdigits"))
assert os.path.isdir("optdigits"), "optdigits フォルダが見つかりません。設定を確認してください。"

## 1. ライブラリとハイパーパラメータ

ネットワークの形と学習設定をまとめて宣言します。ここを書き換えて再実行するだけで、
色々な条件を試せる作りになっています。

- `INPUT_SIZE=64`：入力は 8×8=64 画素
- `HIDDEN_SIZE=[64]*7`：**隠れ層を7つ**重ねた深いネットワーク
- `OUTPUT_SIZE=10`：0〜9 の10クラス
- `LEARNING_RATE` / `EPOCHS` / `DROPOUT_RATE`：学習率・繰り返し回数・ドロップアウト率
- `ACTIVATION`：活性化関数の切替（`relu` / `sigmoid` / `gelu` / `silu`）
- `VISUALIZE_EPOCHS`：この epoch のときに途中経過を可視化する
- `np.random.seed(0)`：再現性のための乱数固定

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt

DATA_DIR = "optdigits"
TRAIN_FILE = os.path.join(DATA_DIR, "optdigits.tra")
TEST_FILE  = os.path.join(DATA_DIR, "optdigits.tes")

INPUT_SIZE = 64
HIDDEN_SIZE = [64, 64, 64, 64, 64, 64, 64]
OUTPUT_SIZE = 10

LEARNING_RATE = 0.01
EPOCHS = 300
DROPOUT_RATE = 0.01

ACTIVATION = "relu"   # "sigmoid" / "relu" / "gelu" / "silu"

SHOW_SAMPLE_COUNT = 100
SHOW_COLS = 10

VISUALIZE_EPOCHS = [0, 1, 10, 50, 100, 200, 299]
VISUALIZE_SAMPLE_INDEX = 0

np.random.seed(0)

## 2. データの読み込みと前処理

`optdigits.tra`（訓練 3823件）と `optdigits.tes`（検証 1797件）を読み込みます。
各行は「64個の画素値(0〜16) + 正解ラベル(0〜9)」です。

- 画素値を `/16.0` して **0〜1 に正規化**（学習を安定させる定番処理）
- ラベルは `one_hot` で「その桁だけ 1、他は 0」の10次元ベクトルに変換
  （例：`3` → `[0,0,0,1,0,0,0,0,0,0]`）。出力の確率分布と比較しやすくするため

In [ ]:
def load_optdigits(path):
    data = np.loadtxt(path, delimiter=",", dtype=np.float32)
    X = data[:, 0:64] / 16.0
    y = data[:, 64].astype(np.int32)
    return X, y

X_train, y_train = load_optdigits(TRAIN_FILE)
X_test, y_test = load_optdigits(TEST_FILE)

def one_hot(y, n):
    t = np.zeros((len(y), n), dtype=np.float32)
    t[np.arange(len(y)), y] = 1.0
    return t

T_train = one_hot(y_train, OUTPUT_SIZE)

## 3. 学習データを見てみる

`visualize_digit_grid` は 64次元のベクトルを 8×8 の画像に戻して並べる可視化関数です
（予測付きの場合、誤分類を赤系で強調表示できます）。

まずは設定値を表示し、**訓練データがどんな数字画像なのか**を100枚ほど眺めてみましょう。
「機械が学ぶ対象」を自分の目で確認しておくのは大切です。

In [ ]:
def visualize_digit_grid(X, y_true, y_pred=None, title="", max_images=100, cols=10):
    if max_images is None:
        max_images = len(X)

    n = min(max_images, len(X))
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.5))

    if rows == 1:
        axes = np.array([axes])

    for i in range(rows * cols):
        ax = axes[i // cols, i % cols]
        ax.axis("off")

        if i >= n:
            continue

        img = X[i].reshape(8, 8)

        if y_pred is None:
            ax.imshow(img, cmap="gray_r", vmin=0.0, vmax=1.0, interpolation="nearest")
            ax.set_title(f"{y_true[i]}", fontsize=10)
        else:
            correct = int(y_true[i]) == int(y_pred[i])

            if correct:
                ax.imshow(img, cmap="gray_r", vmin=0.0, vmax=1.0, interpolation="nearest")
            else:
                ax.imshow(img, cmap="Reds", vmin=0.0, vmax=1.0, interpolation="nearest")

            ax.set_title(
                f"T:{y_true[i]}\nP:{y_pred[i]}",
                fontsize=9,
                color="black" if correct else "red"
            )

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

print("train =", len(X_train))
print("test  =", len(X_test))
print("HIDDEN_SIZE =", HIDDEN_SIZE)
print("LEARNING_RATE =", LEARNING_RATE)
print("DROPOUT_RATE =", DROPOUT_RATE)
print("ACTIVATION =", ACTIVATION)

visualize_digit_grid(
    X_train,
    y_train,
    y_pred=None,
    title="Training data sample",
    max_images=SHOW_SAMPLE_COUNT,
    cols=SHOW_COLS
)

## 4. 活性化関数とその微分

隠れ層に非線形性を与える関数を4種類用意し、`ACTIVATION` で切り替えられるようにします。
逆伝播にはそれぞれの**微分**が必要なので、`*_grad` も対で実装しています。

- **sigmoid**：0〜1 に押し込む。深い層では勾配が小さくなりやすい（勾配消失の主犯）
- **relu**：`max(0,x)`。勾配が消えにくく、深いネットで定番
- **gelu / silu**：より滑らかな新しめの活性化関数

後の実験で `ACTIVATION` を切り替えると、勾配の棒グラフがどう変わるかを比較できます。

In [ ]:
def sigmoid(x):
    x = np.clip(x, -50, 50)
    return 1.0 / (1.0 + np.exp(-x))

def sigmoid_grad(x):
    s = sigmoid(x)
    return s * (1.0 - s)

def relu(x):
    return np.maximum(0, x)

def relu_grad(x):
    return (x > 0).astype(np.float32)

def gelu(x):
    return 0.5 * x * (
        1.0 + np.tanh(
            np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)
        )
    )

def gelu_grad(x):
    u = np.sqrt(2.0 / np.pi) * (x + 0.044715 * x**3)
    tanh_u = np.tanh(u)
    sech2_u = 1.0 - tanh_u**2
    du = np.sqrt(2.0 / np.pi) * (1.0 + 3.0 * 0.044715 * x**2)
    return 0.5 * (1.0 + tanh_u) + 0.5 * x * sech2_u * du

def silu(x):
    return x * sigmoid(x)

def silu_grad(x):
    s = sigmoid(x)
    return s + x * s * (1.0 - s)

def activation(x):
    if ACTIVATION == "sigmoid":
        return sigmoid(x)
    if ACTIVATION == "relu":
        return relu(x)
    if ACTIVATION == "gelu":
        return gelu(x)
    if ACTIVATION == "silu":
        return silu(x)
    raise ValueError(ACTIVATION)

def activation_grad(x):
    if ACTIVATION == "sigmoid":
        return sigmoid_grad(x)
    if ACTIVATION == "relu":
        return relu_grad(x)
    if ACTIVATION == "gelu":
        return gelu_grad(x)
    if ACTIVATION == "silu":
        return silu_grad(x)
    raise ValueError(ACTIVATION)

## 5. 出力・損失・ドロップアウト

- **softmax**：出力の生の値（logits）を、合計1の**確率分布**に変換
- **cross_entropy**：正解クラスの確率が低いほど大きくなる損失（分類の定番）
- **apply_dropout**：学習中だけランダムに一部のユニットを 0 にして、過学習を抑える
  （残したユニットを `1/(1-rate)` 倍して合計の大きさを保つ＝**逆ドロップアウト**）

In [ ]:
def softmax(z):
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)

def cross_entropy(y, t):
    eps = 1e-12
    return -np.sum(t * np.log(y + eps))

def apply_dropout(h, rate):
    if rate <= 0.0:
        return h, None
    mask = (np.random.rand(*h.shape) >= rate).astype(np.float32)
    return h * mask / (1.0 - rate), mask

## 6. 重みの初期化（He初期化）

各層の重みを乱数で初期化します。ReLU 系と相性のよい **He初期化**
（分散が `2/入力数` になるようスケール）を使い、層を重ねても信号が消えたり
爆発したりしにくくしています。バイアスは 0 から始めます。

`layer_sizes` は `[64, 64,64,64,64,64,64,64, 10]`（入力→隠れ7層→出力）になります。

In [ ]:
layer_sizes = [INPUT_SIZE] + HIDDEN_SIZE + [OUTPUT_SIZE]
print("layer_sizes =", layer_sizes)

Ws = []
bs = []

for i in range(len(layer_sizes) - 1):
    in_size = layer_sizes[i]
    out_size = layer_sizes[i + 1]
    W = (np.random.randn(in_size, out_size) * np.sqrt(2.0 / in_size)).astype(np.float32)
    b = np.zeros(out_size, dtype=np.float32)
    Ws.append(W)
    bs.append(b)

## 7. 順伝播 forward

入力から出力までを計算し、あとで逆伝播に使う中間結果（`caches`）も保存します。

各隠れ層で `z = h@W + b` → 活性化 →（学習時のみ）ドロップアウト、を繰り返し、
最後の層で softmax をかけて確率 `y` を得ます。`training=True` のときだけ
ドロップアウトが効く点に注目してください。

In [ ]:
def forward(x, training=False):
    caches = []
    h = x

    for layer in range(len(Ws) - 1):
        z = h @ Ws[layer] + bs[layer]
        h_act = activation(z)

        if training:
            h_drop, mask = apply_dropout(h_act, DROPOUT_RATE)
        else:
            h_drop = h_act
            mask = None

        caches.append({
            "input": h,
            "z": z,
            "h_act": h_act,
            "h": h_drop,
            "dropout_mask": mask
        })

        h = h_drop

    logits = h @ Ws[-1] + bs[-1]
    y = softmax(logits)

    caches.append({
        "input": h,
        "logits": logits,
        "y": y
    })

    return y, caches

## 8. 「1枚が各層を通る様子」を見る可視化

- `vector_to_grid`：ベクトルを画像として見えるよう整形（入力は8×8、出力は1×10）
- `visualize_forward_horizontal`：入力 → 各隠れ層の活性 → 出力（確率）を横一列に並べて表示

学習が進むにつれ、この「流れ」がどう変化するか（各層の活性がどう研ぎ澄まされるか）を
後で epoch ごとに観察します。

In [ ]:
def vector_to_grid(v, width=8):
    v = np.asarray(v)

    if len(v) == 64:
        return v.reshape(8, 8)

    if len(v) == 10:
        return v.reshape(1, 10)

    h = int(np.ceil(len(v) / width))
    padded = np.zeros(h * width, dtype=np.float32)
    padded[:len(v)] = v
    return padded.reshape(h, width)

def visualize_forward_horizontal(epoch, x, true_label):
    y, caches = forward(x, training=False)
    pred_label = int(np.argmax(y))

    panels = [("input", x)]

    for i in range(len(caches) - 1):
        panels.append((f"L{i+1:02d}", caches[i]["h_act"]))

    panels.append(("output", y))

    cols = len(panels)
    fig, axes = plt.subplots(1, cols, figsize=(cols * 1.4, 2.0))

    if cols == 1:
        axes = [axes]

    for ax, (name, v) in zip(axes, panels):
        ax.imshow(vector_to_grid(v), aspect="auto")
        ax.set_title(name, fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(f"epoch={epoch} true={true_label} pred={pred_label}", fontsize=12)
    plt.tight_layout()
    plt.show()

## 9. 勾配の大きさを層ごとに見る（勾配消失の可視化）

このノートの**目玉**です。`collect_gradient_means` は逆伝播をたどり、各層に届く勾配の
平均的な大きさを集めます。`visualize_gradient_bar` はそれを対数スケールの棒グラフにします。

- 棒が長い＝勾配が大きい（よく学習できる）／短い＝勾配が小さい（学習が進みにくい）
- **入力に近い層ほど棒が短くなっていく**なら、それが「勾配消失」です

活性化関数を `sigmoid` にするとこの現象が顕著に、`relu` では緩和されるはずです。

In [ ]:
def collect_gradient_means(x, t):
    y, caches = forward(x, training=False)

    dz = y - t
    last = len(Ws) - 1

    grad_names = ["OUT"]
    grad_means = [float(np.mean(np.abs(dz)))]

    W_old = Ws[last].copy()
    dh = W_old @ dz

    for layer in range(last - 1, -1, -1):
        cache = caches[layer]
        dz_hidden = dh * activation_grad(cache["z"])

        grad_names.append(f"L{layer+1:02d}")
        grad_means.append(float(np.mean(np.abs(dz_hidden))))

        W_old = Ws[layer].copy()
        dh = W_old @ dz_hidden

    grad_names = grad_names[::-1]
    grad_means = grad_means[::-1]

    return grad_names, grad_means

def visualize_gradient_bar(epoch, x, t):
    names, means = collect_gradient_means(x, t)

    means = np.array(means, dtype=np.float64)
    eps = 1e-20
    log_means = np.log10(means + eps)

    min_log = np.floor(np.min(log_means))
    bar_lengths = log_means - min_log

    plt.figure(figsize=(10, 3.2))
    plt.bar(names, bar_lengths)

    plt.title(f"epoch={epoch} gradient mean, larger bar = larger gradient")
    plt.xlabel("Layer")
    plt.ylabel("Relative log scale")

    for i, v in enumerate(bar_lengths):
        plt.text(
            i,
            v + 0.05,
            f"{means[i]:.1e}",
            ha="center",
            va="bottom",
            fontsize=8
        )

    plt.grid(axis="y")
    plt.tight_layout()
    plt.show()

## 10. 学習ループ

1サンプルずつ（オンライン学習）、順伝播 → 交差エントロピー → 誤差逆伝播 → SGD更新を
`EPOCHS` 回繰り返します。`VISUALIZE_EPOCHS` に指定した epoch では、途中経過として
「forward の様子」と「層ごとの勾配」を描画します。

**実行には数分かかることがあります**（`EPOCHS=300`）。まずはそのまま流して、
勾配の棒グラフや活性の変化が epoch とともにどう動くかを眺めてください。

In [ ]:
for epoch in range(EPOCHS):
    total_loss = 0.0
    indices = np.random.permutation(len(X_train))

    for idx in indices:
        x = X_train[idx]
        t = T_train[idx]

        y, caches = forward(x, training=True)
        loss = cross_entropy(y, t)
        total_loss += loss

        dz = y - t
        last = len(Ws) - 1

        h_prev = caches[last]["input"]
        dW = np.outer(h_prev, dz)
        db = dz.copy()

        W_old = Ws[last].copy()
        dh = W_old @ dz

        Ws[last] -= LEARNING_RATE * dW
        bs[last] -= LEARNING_RATE * db

        for layer in range(last - 1, -1, -1):
            cache = caches[layer]

            if cache["dropout_mask"] is not None:
                dh = dh * cache["dropout_mask"] / (1.0 - DROPOUT_RATE)

            dz = dh * activation_grad(cache["z"])

            h_prev = cache["input"]
            dW = np.outer(h_prev, dz)
            db = dz.copy()

            W_old = Ws[layer].copy()
            dh = W_old @ dz

            Ws[layer] -= LEARNING_RATE * dW
            bs[layer] -= LEARNING_RATE * db

    avg_loss = total_loss / len(X_train)

    if epoch % 50 == 0:
        print(f"epoch={epoch}, loss={avg_loss:.6f}")

    if epoch in VISUALIZE_EPOCHS:
        sample_x = X_train[VISUALIZE_SAMPLE_INDEX]
        sample_y = y_train[VISUALIZE_SAMPLE_INDEX]
        sample_t = T_train[VISUALIZE_SAMPLE_INDEX]

        visualize_forward_horizontal(epoch, sample_x, sample_y)
        visualize_gradient_bar(epoch, sample_x, sample_t)

## 11. 評価（テストデータで予測し、結果を可視化）

学習済みネットワークで検証データ 1797件を予測し、**混同行列**（どの数字をどの数字と
間違えたかの表）を作ります。あわせて `visualize_digit_grid` で、評価サンプルを
`T:正解 / P:予測` 付きで表示し、誤分類は色で強調します。

「4 と 9」「3 と 8」のように、人間でも紛らわしいペアで間違えやすい傾向が見えるはずです。

In [ ]:
def predict_proba(x):
    y, _ = forward(x, training=False)
    return y

def predict(x):
    y = predict_proba(x)
    return int(np.argmax(y)), y

test_pred = np.zeros(len(X_test), dtype=np.int32)
confusion = np.zeros((OUTPUT_SIZE, OUTPUT_SIZE), dtype=np.int32)

for i in range(len(X_test)):
    true_label = int(y_test[i])
    pred_label, _ = predict(X_test[i])

    test_pred[i] = pred_label
    confusion[true_label, pred_label] += 1

visualize_digit_grid(
    X_test,
    y_test,
    y_pred=test_pred,
    title="Evaluation result sample  T=true / P=pred",
    max_images=SHOW_SAMPLE_COUNT,
    cols=SHOW_COLS
)

## 12. 指標（accuracy / precision / recall / F1）

混同行列から評価指標を計算します。

- **accuracy**：全体の正解率
- **precision（適合率）**：そのクラスと予測したうち、本当に正解だった割合
- **recall（再現率）**：本当にそのクラスのうち、正しく当てられた割合
- **F1**：precision と recall の調和平均（バランス指標）
- **macro**：全クラスの単純平均

数字ごとの得意・不得意（precision/recall の偏り）も読み取れます。

In [ ]:
print("\n=== confusion matrix ===")
print(confusion)

accuracy = np.trace(confusion) / np.sum(confusion)

macro_precision = 0.0
macro_recall = 0.0
macro_f1 = 0.0

print("\n=== per class metrics ===")

for c in range(OUTPUT_SIZE):
    tp = confusion[c, c]
    fp = np.sum(confusion[:, c]) - tp
    fn = np.sum(confusion[c, :]) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    macro_precision += precision
    macro_recall += recall
    macro_f1 += f1

    print(
        f"{c}: "
        f"precision={precision:.4f}, "
        f"recall={recall:.4f}, "
        f"f1={f1:.4f}"
    )

macro_precision /= OUTPUT_SIZE
macro_recall /= OUTPUT_SIZE
macro_f1 /= OUTPUT_SIZE

print("\n=== overall ===")
print(f"accuracy        = {accuracy:.4f}")
print(f"macro precision = {macro_precision:.4f}")
print(f"macro recall    = {macro_recall:.4f}")
print(f"macro f1        = {macro_f1:.4f}")

## やってみよう（実験のヒント）

設定を変えて再実行し、可視化がどう変わるかを比べてください。

- **勾配消失を体感**：`ACTIVATION` を `"relu"` → `"sigmoid"` に変え、勾配の棒グラフが
  入力側の層でどれだけ短くなるか（勾配が消えるか）を比較
- **層の深さ**：`HIDDEN_SIZE` を `[64, 64]`（浅い）や `[64]*12`（さらに深い）にして
  精度と勾配の変化を見る
- **学習率**：`LEARNING_RATE` を `0.001` / `0.05` に変えて収束の様子を比較
- **ドロップアウト**：`DROPOUT_RATE` を `0.0` / `0.3` にして過学習・精度の違いを観察
- **活性化関数**：`gelu` / `silu` に切り替えて精度と安定性を比べる

> 変更後は「ランタイム > すべてのセルを実行」で最初から流し直してください。